# M7 PTCST ablations
Compare forecast value, optimizer value and cost-awareness using the locked PTCST seed-7 forecast run.

In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = _BOOTSTRAP_REPO / 'scripts' / 'colab_bootstrap.py'
if not _BOOTSTRAP_SCRIPT.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/5ebae86/scripts/colab_bootstrap.py'
    urllib.request.urlretrieve(raw, '/content/colab_bootstrap.py')
    _BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


In [ ]:
# Run the locked ablation ladder with visible diagnostics.
import subprocess, sys
ptcst_run = WORKSPACE / 'runs' / 'v3_ptcst_seed_sweep' / 'seed_7'
ablation_run = WORKSPACE / 'runs' / 'v3_ptcst_ablations'
if not (ptcst_run / 'forecasts.parquet').exists():
    raise FileNotFoundError('PTCST seed-7 forecast is missing. Complete Notebook 04 first.')
cmd = [sys.executable, str(REPO / 'scripts' / 'run_v3_ptcst_ablations.py'), '--data-root', str(DATA_ROOT), '--forecast-run', str(ptcst_run), '--run-dir', str(ablation_run)]
result = subprocess.run(cmd, check=False, capture_output=True, text=True)
if result.stdout: print(result.stdout)
if result.stderr: print('STDERR:\n' + result.stderr)
if result.returncode != 0: raise RuntimeError(f'PTCST ablations failed: {result.stderr or result.stdout}')
import pandas as pd
ablation_summary = pd.read_parquet(ablation_run / 'portfolio_metrics_summary.parquet')
display(ablation_summary)


In [ ]:
# Persist ablation outputs to Drive.
from shutil import copytree
drive_runs = Path('/content/drive/MyDrive/kltn/runs')
drive_runs.mkdir(parents=True, exist_ok=True)
copytree(ablation_run, drive_runs / 'v3_ptcst_ablations', dirs_exist_ok=True)
print('M7 artifacts synced to:', drive_runs / 'v3_ptcst_ablations')
